# Train a single-task regression model from scratch

In this tutorial, we train a single-task convolutional regression model to predict total coverage over DNase-seq peaks, starting from ENCODE DNase-seq read coverage file.

In [1]:
import os
import numpy as np
import pandas as pd
import torch

from plotnine import *
%matplotlib inline

from grelu.io.bed import read_bed
from  grelu.data.preprocess import split
from grelu.lightning import LightningModel
from grelu.data.dataset import BigWigSeqDataset

import wandb
wandb.login(host='https://api.wandb.ai', relogin=True)
os.environ["WANDB_NOTEBOOK_NAME"] = "/code/github/gReLU-applications/VEP_benchmark/2_train_GM12878_DNase.ipynb"
project_name='GM12878_dnase'

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: avantikalal (grelu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Model parameters

In [2]:
model_params = {
    'model_type':'DilatedConvModel',
    'crop_len':(2114-1000)//2,
    'n_tasks':1,
    'channels':512,
    'n_conv':9,
}

train_params = {
    'task':'regression',
    'loss': 'mse', 
    'logger':'wandb',
    'lr':1e-4,
    'batch_size':512,
    'max_epochs':15,
    'devices':0,
    'num_workers':16,
    'save_dir':'.',
    'checkpoint': {"monitor":"val_pearson", "mode":"max", "save_last":True}
}

## Set up sweep

In [3]:
sweep_params = {
    'method': 'grid',
    'name': project_name,
    
    'metric': {
        'goal': 'minimize', 
        'name': 'validation_loss',
        },

    'parameters': {
        'rc':{
            'distribution': 'categorical',
            'values':[False, True],
        },
        'max_seq_shift':{
            'distribution': 'categorical',
            'values':[0, 1, 3],
        },
        'max_pair_shift':{
            'distribution': 'categorical',
            'values':[0, 10, 50, 100],
        },
    }
}

In [4]:
def build_and_train_model(model_params=model_params, train_params=train_params):
    
    run = wandb.init(dir='.')

    artifact = run.use_artifact('GM12878_dnase/dataset:latest')
    dir = artifact.download()
    intervals = read_bed(os.path.join(dir, "intervals.bed"))

    train, val, test = split(intervals, val_chroms=["chr10"], test_chroms=["chr11"])
    train_dataset = BigWigSeqDataset(
        intervals = train, 
        bw_files=["ENCFF093VXI.bigWig"],
        label_len=1000, 
        label_aggfunc="sum",
        label_transform_func=np.log1p,
        rc=wandb.config["rc"],
        max_seq_shift=wandb.config["max_seq_shift"], 
        max_pair_shift=wandb.config["max_pair_shift"],
        augment_mode="random", 
        seed=0, 
        genome='hg38'
    )
    val_dataset = BigWigSeqDataset(
        intervals = val, 
        bw_files=["ENCFF093VXI.bigWig"],
        label_len=1000, label_aggfunc="sum", 
        label_transform_func=np.log1p, genome='hg38'
    )

    model = LightningModel(model_params=model_params, train_params=train_params)
    model.train_on_dataset(train_dataset, val_dataset)

## Run sweep

In [5]:
sweep_id = wandb.sweep(sweep=sweep_params, project=project_name) 

Create sweep with ID: udnpa77q
Sweep URL: https://wandb.ai/grelu/GM12878_dnase/sweeps/udnpa77q


In [ ]:
wandb.agent(sweep_id, function=build_and_train_model)

wandb: Agent Starting Run: dcq2sg8p with config:
wandb: 	max_pair_shift: 0
wandb: 	max_seq_shift: 0
wandb: 	rc: False


wandb:   1 of 1 files downloaded.  


Selecting training samples
Keeping 390473 intervals


Selecting validation samples
Keeping 21987 intervals


Selecting test samples
Keeping 22595 intervals
Final sizes: train: (390473, 3), val: (21987, 3), test: (22595, 3)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/conda/lib/python3.11/site-packages/pytorch_lightning/loggers/wandb.py:396: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Validation DataLoader 0: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 43/43 [00:09<00:00,  4.44it/s]


/opt/conda/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The variance of predictions or target is close to zero. This can cause instability in Pearson correlationcoefficient, leading to wrong results. Consider re-scaling the input if possible or computing using alarger dtype (currently using torch.float32).


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val_loss          │     18.51033592224121     │
│          val_mse          │    18.508310317993164     │
│        val_pearson        │   -0.05173555389046669    │
└───────────────────────────┴───────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name         | Type             | Params | Mode 
----------------------------------------------------------
0 | model        | DilatedConvModel | 6.3 M  | train
1 | loss         | MSELoss          | 0      | train
2 | activation   | Identity         | 0      | train
3 | val_metrics  | MetricCollection | 0      | train
4 | test_metrics | MetricCollection | 0      | train
5 | transform    | Identity         | 0      | train
----------------------------------------------------------
6.3 M     Trainable params
0         Non-trainable params
6.3 M     Total params
25.358    Total estimated model params size (MB)


/opt/conda/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The variance of predictions or target is close to zero. This can cause instability in Pearson correlationcoefficient, leading to wrong results. Consider re-scaling the input if possible or computing using alarger dtype (currently using torch.float32).


Epoch 0: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 763/763 [07:41<00:00,  1.65it/s, v_num=sg8p, train_loss_step=0.591]
Validation: |                                                                                                                                                                                                        | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 763/763 [07:42<00:00,  1.65it/s, v_num=sg8p, train_loss_step=1.000, train_loss_epoch=1.040]
Validation: |                                                                                                                                                                                                        | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████████████████████████████████████